In [3]:
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

df = pd.read_csv("../data/raw/day.csv", parse_dates=["dteday"])
assert not df["dteday"].isna().any()
assert not df["dteday"].duplicated().any()
y = df.sort_values("dteday").set_index("dteday")["cnt"]
y = y.asfreq("D")
assert len(y) == 731 and not y.isna().any()
assert np.isfinite(y).all() and (y >= 0).all()

assert (y % 1 == 0).all()

train = y.loc["2011-01-01":"2012-12-17"]

test = y.loc["2012-12-18":"2012-12-31"]

assert len(train) == 717 and len(test) == 14

 

model = SARIMAX(train, order=(1, 1, 1),

               seasonal_order=(1, 0, 1, 7), trend="n")

fitted = model.fit(disp=False, maxiter=500)

print("Converged:", fitted.mle_retvals.get("converged"))

fc = fitted.get_forecast(steps=14)

ci = fc.conf_int(alpha=0.05)



# Repeat the FINAL TRAINING week twice; never use test counts.

baseline = np.tile(train.iloc[-7:].to_numpy(), 2)

results = pd.DataFrame({

   "actual_cnt": test, "predicted_cnt": fc.predicted_mean,

   "baseline_cnt": baseline, "lower_95": ci.iloc[:, 0],

   "upper_95": ci.iloc[:, 1]})

assert results.index.equals(test.index)

results["absolute_error"] = abs(results.actual_cnt-results.predicted_cnt)

results.rename_axis("date").to_csv("test_predictions.csv")

Converged: True
